# 🧬 Molecular Docking Pipeline — Data Generation

**Notebook:** `01_docking_pipeline.ipynb`  
**Project:** Joint Molecular Optimization for Virtual Screening  
**Dataset:** DUD-E (Database of Useful Decoys: Enhanced)

---

## Overview

This notebook implements the **first stage** of the research pipeline: generating docking scores for all active compounds and decoys against target protein receptors using **AutoDock SMINA**.

The output is a structured CSV per target containing:
- `SMILES` — molecular structure string
- `docking_score` — predicted binding affinity (kcal/mol, more negative = stronger binding)

This CSV feeds directly into `02_ml_pipeline.ipynb` for machine learning model training.

### Pipeline Flow
```
DUD-E SDF files
    ↓ split_sdf()        — Split multi-molecule SDF into individual files
    ↓ selector()         — Balance actives vs. decoys (1:1 ratio)
    ↓ SMINA docking      — Compute binding affinities for each molecule
    ↓ sdf_to_smiles()    — Convert structures to SMILES strings
    ↓ DataFrame + CSV    — Save docking_data.csv per target
```

### Targets Used in This Study
| Target | Description |
|--------|-------------|
| `parp1` | Poly(ADP-ribose) polymerase 1 (cancer) |
| `ital` | Integrin alpha-L (immune signaling) |
| `mapk2` | Mitogen-activated protein kinase 2 |
| `tgfr1` | TGF-beta receptor type 1 |

> **Runtime Note:** This notebook was designed for **Google Colab** with Google Drive mounted at `/content/drive/MyDrive/COLAB_DATA/`. Adjust paths for your environment.

---

## 1. Environment Setup & Library Installation

Installs and imports all required packages for this notebook:

| Library | Purpose |
|---|---|
| `rdkit` | Cheminformatics — SMILES parsing, fingerprints, molecular descriptors |
| `openbabel` | File format conversion (PDB/SDF → PDBQT) required by SMINA |
| `scikit-learn` | Machine learning utilities (used in downstream notebooks) |
| `pandas` | Data manipulation and CSV I/O |
| `subprocess` | Calling SMINA as an external command-line process |
| `re` | Regex parsing of SMINA stdout to extract docking scores |

> **Note:** `smina.static` must be present in your working directory or accessible via the path configured in Cell 3. Download it from [SourceForge](https://sourceforge.net/projects/smina/).

In [ ]:
# ── Install system dependencies ──────────────────────────────────────────────
# Run these lines once when setting up the environment.
# rdkit: core cheminformatics library
# openbabel: handles PDB → PDBQT conversion needed by SMINA
# smina.static: the molecular docking executable (downloaded separately)

!apt-get install -y openbabel
!pip install py3Dmol rdkit-pypi scikit-learn pandas

# Download SMINA docking engine and make it executable
!wget https://downloads.sourceforge.net/project/smina/smina.static -O smina.static
!chmod +x smina.static

# ── Core Python imports ───────────────────────────────────────────────────────
import csv
import os
import subprocess
import re
import pickle

import pandas as pd

# RDKit: molecular manipulation
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw, DataStructs, RDConfig, rdBase
from rdkit.Chem.Draw import IPythonConsole

# scikit-learn: used for downstream model training
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"RDKit version: {rdBase.rdkitVersion}")

## 2. Data Directory Scanner

Scans the top-level DUD-E data directory to enumerate all available target folders.

Each target folder in the DUD-E dataset contains:
- `receptor.pdb` — 3D protein structure
- `actives_final.sdf` — Known active compounds (positive class)
- `decoys_final.sdf` — Property-matched decoys (negative class)
- `crystal_ligand.mol2` — Co-crystallized ligand used to define the docking search box

**Output:** A list of folder names (target names) to iterate over in the docking loop.

In [ ]:
def list_folders(mega_folder_path):
    """
    Returns the names of all subdirectories within mega_folder_path.
    Each subdirectory represents one DUD-E target (e.g., 'parp1', 'ital').

    Parameters
    ----------
    mega_folder_path : str
        Root path containing per-target DUD-E directories.

    Returns
    -------
    list[str]
        Names of subdirectories found at the path.
    """
    try:
        all_items = os.listdir(mega_folder_path)
        folders = [item for item in all_items
                   if os.path.isdir(os.path.join(mega_folder_path, item))]
        return folders
    except Exception as e:
        print(f"Error scanning directory: {e}")
        return []


# ── Configuration ─────────────────────────────────────────────────────────────
# Update this path to point to your DUD-E data root.
mega_folder_path = 'drive/MyDrive/COLAB_DATA/all/all'
folders = list_folders(mega_folder_path)

print(f"Found {len(folders)} target folders:")
for f in sorted(folders):
    print(f"  - {f}")

## 3. Utility Functions

Reusable helper functions used throughout the docking pipeline:

| Function | Description |
|---|---|
| `read_smiles(file_path)` | Reads a plain-text SMILES file (one SMILES per line) into a Python list |
| `run_smina(...)` | Wrapper that invokes SMINA via `subprocess` and handles stdout/stderr |
| `check_file_exists(filepath)` | Asserts a file is present before proceeding; raises `FileNotFoundError` on failure |

In [ ]:
def read_smiles(file_path):
    """
    Read a plain-text SMILES file where each line contains one SMILES string.

    Parameters
    ----------
    file_path : str
        Path to the SMILES text file.

    Returns
    -------
    list[str]
        Stripped SMILES strings, one per line.
    """
    with open(file_path, 'r') as file:
        lines = file.readlines()
    return [line.strip() for line in lines]


def run_smina(receptor_filepath, ligand_pdbqt, output_file, log_file):
    """
    Run a SMINA docking calculation and save the docked pose and log.

    Parameters
    ----------
    receptor_filepath : str
        Path to the receptor in PDBQT format.
    ligand_pdbqt : str
        Path to the ligand in PDBQT format.
    output_file : str
        Path where the docked pose output (PDBQT) will be saved.
    log_file : str
        Path where the SMINA log (including scores) will be written.
    """
    command = [
        smina_path,
        '-r', receptor_filepath,
        '-l', ligand_pdbqt,
        '--out', output_file,
        '--log', log_file
    ]
    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"SMINA error: {result.stderr}")
    else:
        print(f"Docking complete → {output_file}")
        print(f"SMINA output:\n{result.stdout}")


def check_file_exists(filepath):
    """
    Verify that a file exists at the given path.

    Parameters
    ----------
    filepath : str
        Absolute or relative path to check.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    """
    if os.path.exists(filepath):
        print(f"✔ Found: {filepath}")
    else:
        raise FileNotFoundError(f"✘ Missing: {filepath}")

## 4. File Path Checker

Quick diagnostic utility to confirm that the Google Drive mount and data directories are accessible before running the docking loop.

Run this cell first after mounting Drive to catch path issues early.

In [ ]:
def check_path_exists(path):
    """
    Print whether a directory or file path exists on disk.

    Parameters
    ----------
    path : str
        Path to check (file or directory).

    Returns
    -------
    bool
        True if the path exists, False otherwise.
    """
    if os.path.exists(path):
        print(f"✔ Path exists: '{path}'")
        return True
    else:
        print(f"✘ Path NOT found: '{path}'")
        return False


# ── Diagnostic checks ─────────────────────────────────────────────────────────
# Verify Drive mount and critical data folders are accessible.
check_path_exists("/content/drive/MyDrive/COLAB_DATA")
check_path_exists("/content/drive/MyDrive/COLAB_DATA/all/all")

## 5. SDF Splitter — Individual Molecule Files

DUD-E provides actives and decoys as **multi-molecule SDF files**. SMINA requires each ligand as a separate PDBQT file.

This cell:
1. Reads every molecule from the input SDF using RDKit's `SDMolSupplier`
2. Writes each molecule to its own `.sdf` file
3. Converts each individual SDF → PDBQT using Open Babel (`obabel`)

**Run this cell once per target** for both `actives_final.sdf` and `decoys_final.sdf`.

> **Output directory structure:**
> ```
> tysy/
> ├── split_molecules/        ← actives as individual PDBQT files
> └── decoy_split_molecules/  ← decoys as individual PDBQT files
> ```

In [ ]:
def split_sdf(input_sdf, output_dir):
    """
    Split a multi-molecule SDF file into individual PDBQT files.

    Each molecule is first written as a temporary SDF, then converted to PDBQT
    using Open Babel. Molecules that fail to parse are silently skipped.

    Parameters
    ----------
    input_sdf : str
        Path to the multi-molecule SDF file (e.g., actives_final.sdf).
    output_dir : str
        Directory where individual PDBQT files will be saved.
        Created automatically if it does not exist.

    Side Effects
    ------------
    Writes `molecule_N.pdbqt` for each successfully parsed molecule.
    Prints a summary count on completion.
    """
    os.makedirs(output_dir, exist_ok=True)
    suppl = Chem.SDMolSupplier(input_sdf)

    count = 0
    for i, mol in enumerate(suppl):
        if mol is None:
            continue  # skip unparseable entries

        pdbqt_path = os.path.join(output_dir, f'molecule_{i + 1}.pdbqt')
        sdf_path   = pdbqt_path.replace('.pdbqt', '.sdf')

        # Write temporary SDF for this molecule
        writer = Chem.SDWriter(sdf_path)
        writer.write(mol)
        writer.close()

        # Convert SDF → PDBQT using Open Babel
        subprocess.run(f"obabel '{sdf_path}' -O '{pdbqt_path}'", shell=True)
        count += 1

    print(f"✔ Split complete: {count} molecules written to '{output_dir}'")


# ── Example usage ─────────────────────────────────────────────────────────────
# Update target name and paths as needed.
TARGET = 'tysy'  # Change this to your target folder name
BASE   = f'/content/drive/MyDrive/COLAB_DATA/all/all/{TARGET}'

split_sdf(
    input_sdf  = os.path.join(BASE, 'actives_final.sdf'),
    output_dir = os.path.join(BASE, 'split_molecules')
)

split_sdf(
    input_sdf  = os.path.join(BASE, 'decoys_final.sdf'),
    output_dir = os.path.join(BASE, 'decoy_split_molecules')
)

## 6. Dataset Balancer — Active / Decoy Selector

DUD-E datasets are **heavily imbalanced**: decoys typically outnumber actives by 50:1 or more. Training on unbalanced data biases models toward predicting everything as inactive.

This cell equalizes the dataset by:
1. Counting PDBQT files in both the actives and decoys directories
2. Taking the **smaller count** as the limit for both sets
3. Returning a pair of file lists of equal length (truncated from the larger set)

The 1:1 active-to-decoy ratio used here is a deliberate design choice to avoid class imbalance bias in the downstream ML models.

In [ ]:
def get_sdf_files(folder_path):
    """
    Return a sorted list of absolute paths for all .sdf files in folder_path.

    Parameters
    ----------
    folder_path : str
        Directory to scan.

    Returns
    -------
    list[str]
        Sorted absolute paths of .sdf files found.
    """
    return sorted([
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.endswith('.sdf')
    ])


def balance_dataset(actives_dir, decoys_dir):
    """
    Balance the active and decoy file lists to an equal 1:1 ratio.

    Counts .sdf files in both directories and truncates the larger list
    to match the smaller one, preventing class imbalance during training.

    Parameters
    ----------
    actives_dir : str
        Directory containing split active compound SDF files.
    decoys_dir : str
        Directory containing split decoy compound SDF files.

    Returns
    -------
    tuple[list[str], list[str]]
        (active_files, decoy_files) — two lists of equal length.
    """
    active_files = get_sdf_files(actives_dir)
    decoy_files  = get_sdf_files(decoys_dir)

    n_active = len(active_files)
    n_decoy  = len(decoy_files)
    n_balanced = min(n_active, n_decoy)

    print(f"Actives available : {n_active}")
    print(f"Decoys  available : {n_decoy}")
    print(f"Using {n_balanced} from each → balanced 1:1 dataset")

    return active_files[:n_balanced], decoy_files[:n_balanced]


# ── Balance dataset for current target ────────────────────────────────────────
a1_files, d_files = balance_dataset(
    actives_dir = os.path.join(BASE, 'split_molecules'),
    decoys_dir  = os.path.join(BASE, 'decoy_split_molecules')
)

## 7. Main Docking Loop — Training Data Generation

The core data-generation loop. For each molecule in the balanced active + decoy set:

1. **Convert receptor** — PDB → PDBQT (once per target via Open Babel)
2. **Run SMINA** — dock each molecule against the receptor using the crystal ligand to auto-define the search box
3. **Parse docking score** — extract the first-mode binding affinity from SMINA's stdout with regex
4. **Extract SMILES** — convert the SDF back to a canonical SMILES string using RDKit
5. **Accumulate results** — build a DataFrame and save to `docking_data.csv`

### SMINA Parameters
| Parameter | Value | Explanation |
|---|---|---|
| `--autobox_ligand` | `crystal_ligand.mol2` | Defines search box around co-crystallized pose |
| `--autobox_add` | `8` Å | Expands box by 8 Å in each direction |
| Output | `redock_N.pdbqt` | Docked pose for compound N |

> **Compute time:** This loop is the bottleneck of the pipeline. On a CPU-only machine, expect ~30–120 seconds per molecule. For large datasets, use the **Segmented Version** in Cell 8 instead.

In [ ]:
def sdf_to_smiles(sdf_path):
    """
    Extract the first molecule from an SDF file and return its canonical SMILES.

    Parameters
    ----------
    sdf_path : str
        Path to a single-molecule SDF file.

    Returns
    -------
    str
        Canonical SMILES string, or None if parsing fails.
    """
    suppl = Chem.SDMolSupplier(sdf_path)
    for mol in suppl:
        if mol is not None:
            return Chem.MolToSmiles(mol)
    return None


# ── Target configuration ──────────────────────────────────────────────────────
# Set `target` to the folder name of the DUD-E target you want to process.
target = TARGET  # defined in Cell 5

drive_path  = "/content/drive/MyDrive/COLAB_DATA/all/all/"
working_dir = os.path.join(drive_path, target)

receptor_path = os.path.join(working_dir, "receptor.pdb")
actives_path  = os.path.join(working_dir, "actives_final.sdf")
decoys_path   = os.path.join(working_dir, "decoys_final.sdf")
crystal       = os.path.join(working_dir, "crystal_ligand.mol2")
smina_path    = os.path.join(working_dir, 'smina.static')

# Validate that all required files are present before starting
for path in [receptor_path, actives_path, decoys_path, crystal, smina_path]:
    check_file_exists(path)

# ── Convert receptor PDB → PDBQT (once per target) ───────────────────────────
receptor_pdbqt = os.path.join(working_dir, 'receptor.pdbqt')
try:
    subprocess.run(
        f"obabel '{receptor_path}' -xr -O '{receptor_pdbqt}'",
        shell=True, check=True
    )
    print("✔ Receptor converted to PDBQT")
except subprocess.CalledProcessError as e:
    print(f"Open Babel conversion failed: {e}")

# ── Main docking loop ─────────────────────────────────────────────────────────
# Iterates over the balanced list of actives + decoys.
# For each molecule: run SMINA, parse score, extract SMILES.
molecule_paths = a1_files + d_files
data = []  # accumulate [SMILES, docking_score] rows

for i, molecule_path in enumerate(molecule_paths):
    if not os.path.exists(molecule_path):
        print(f"[SKIP] File not found: {molecule_path}")
        continue

    redock_pdbqt = os.path.join(working_dir, f'redock_{i + 1}.pdbqt')

    # Build the SMINA command
    smina_command = (
        f"{smina_path} "
        f"-r {receptor_pdbqt} "
        f"-l {molecule_path} "
        f"--autobox_ligand {crystal} "
        f"--autobox_add 8 "
        f"-o {redock_pdbqt}"
    )

    docking_score = None
    try:
        result = subprocess.run(
            smina_command, capture_output=True, text=True, shell=True
        )
        # Parse the binding affinity for pose mode 1 from SMINA stdout.
        # SMINA outputs a table: "  1  <affinity>  ..."
        match = re.search(r"^1\s+([-.\d]+)", result.stdout, re.MULTILINE)
        if match:
            docking_score = float(match.group(1))
            print(f"[{i+1}/{len(molecule_paths)}] Score: {docking_score:.3f} kcal/mol")
        else:
            print(f"[{i+1}/{len(molecule_paths)}] Score not found in SMINA output")

    except subprocess.CalledProcessError as e:
        print(f"SMINA failed for {molecule_path}: {e.stderr}")

    smiles = sdf_to_smiles(molecule_path)
    data.append([smiles, docking_score])

# ── Build and save DataFrame ──────────────────────────────────────────────────
df = pd.DataFrame(data, columns=["SMILES", "docking_score"])
df = df.dropna(subset=["docking_score"])  # remove rows where docking failed

csv_path = os.path.join(working_dir, "docking_data.csv")
df.to_csv(csv_path, index=False)

print(f"\n✔ Docking complete. {len(df)} molecules saved → {csv_path}")
print(df.describe())

## 8. Segmented Docking Loop (Large Datasets)

For large compound libraries where the full docking loop may be interrupted by Colab timeouts or memory limits, this **segmented version** allows you to run the docking in chunks and resume from where you left off.

### How to Use
1. Run **Cell 7 first** with just 1–2 molecules to confirm SMINA is working correctly
2. Stop the cell early
3. Define `segment_index` and slice `molecule_paths` accordingly
4. Results accumulate in the existing `data` list and can be merged with prior runs

### Key Difference From Cell 7
- Output filenames include `segment_index` to avoid overwriting redocked PDBQT files from other runs
- You control exactly which slice of `molecule_paths` to process

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# `segment_index` identifies this batch (0, 1, 2, …) for output file naming.
# `segment` is the slice of molecule_paths you want to dock in this run.
# Example: to process molecules 200–400, use molecule_paths[200:400].

segment_index = 0                  # Change for each batch
segment       = molecule_paths     # Process all molecules; slice to sub-batch if needed

# ── Segmented docking loop ────────────────────────────────────────────────────
# Functionally identical to the main loop in Cell 7, but filenames include the
# segment index so successive runs do not overwrite each other's output files.

for i, molecule_path in enumerate(segment):
    if not os.path.exists(molecule_path):
        print(f"[SKIP] Not found: {molecule_path}")
        continue

    # Unique output name per segment and molecule index
    redock_pdbqt = os.path.join(
        working_dir, f'redock_seg{segment_index}_{i + 1}.pdbqt'
    )

    smina_command = (
        f"{smina_path} "
        f"-r {receptor_pdbqt} "
        f"-l {molecule_path} "
        f"--autobox_ligand {crystal} "
        f"--autobox_add 8 "
        f"-o {redock_pdbqt}"
    )

    docking_score = None
    try:
        result = subprocess.run(
            smina_command, capture_output=True, text=True, shell=True
        )
        match = re.search(r"^1\s+([-.\d]+)", result.stdout, re.MULTILINE)
        if match:
            docking_score = float(match.group(1))
            print(f"  [{i+1}] Score: {docking_score:.3f} kcal/mol")
        else:
            print(f"  [{i+1}] Score not found")

    except subprocess.CalledProcessError as e:
        print(f"  [{i+1}] SMINA failed: {e.stderr}")

    data.append([molecule_path, docking_score])

print(f"\nSegment {segment_index} complete. {len(data)} total rows accumulated.")